In [30]:
import os
import cv2
import numpy as np
from random import randint, uniform, seed, random
seed(42)  # For reproducibility of random numbers

# Set your folder paths
input_folder = "E:/Coins/Greyscaled Coins/Chinese Yuan"  # Replace with your input folder path
output_folder = "E:/Coins/Augmented Coins/Chinese Yuan"  # Replace with your output folder path

# Create output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Function to apply random transformations
def augment_image(image):
    height, width = image.shape[:2]

    # Randomly resize the image
    scale = uniform(0.8, 1.2)  # Resize factor between 80% and 120%
    new_width, new_height = int(width * scale), int(height * scale)
    resized = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_LINEAR)

    # Randomly rotate the image
    angle = randint(-30, 30)  # Rotate between -30 to 30 degrees
    center = (new_width // 2, new_height // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1)
    rotated = cv2.warpAffine(resized, rotation_matrix, (new_width, new_height))

    transformed = rotated

    # Apply translation with a 30% chance
    if random() < 0.3:
        max_trans_x = int(0.2 * new_width)  # Max translation: 20% of width
        max_trans_y = int(0.2 * new_height)  # Max translation: 20% of height
        trans_x = randint(-max_trans_x, max_trans_x)
        trans_y = randint(-max_trans_y, max_trans_y)
        translation_matrix = np.float32([[1, 0, trans_x], [0, 1, trans_y]])
        transformed = cv2.warpAffine(transformed, translation_matrix, (new_width, new_height))

    # Apply perspective transformation with a 30% chance
    if random() < 0.3:
        margin = int(0.1 * min(new_width, new_height))  # 10% margin for perspective
        src_points = np.float32([
            [randint(0, margin), randint(0, margin)],
            [new_width - randint(0, margin), randint(0, margin)],
            [randint(0, margin), new_height - randint(0, margin)],
            [new_width - randint(0, margin), new_height - randint(0, margin)]
        ])
        dst_points = np.float32([
            [0, 0],
            [new_width, 0],
            [0, new_height],
            [new_width, new_height]
        ])
        perspective_matrix = cv2.getPerspectiveTransform(src_points, dst_points)
        transformed = cv2.warpPerspective(transformed, perspective_matrix, (new_width, new_height))

    # Adjust brightness and contrast
    brightness_offset = uniform(-30, 30)  # Brightness change
    contrast_factor = uniform(0.8, 1.2)  # Contrast scale
    augmented = np.clip(transformed * contrast_factor + brightness_offset, 0, 255).astype(np.uint8)

    return augmented

# Main function to process all images
def process_images(input_folder, output_folder, num_augmentations=20):
    for filename in os.listdir(input_folder):
        input_path = os.path.join(input_folder, filename)

        # Check if file is an image
        if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        # Read the image in grayscale
        image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            print(f"Could not read {filename}. Skipping.")
            continue

        base_filename = os.path.splitext(filename)[0]

        # Generate and save unique augmentations
        saved_images = set()  # Track unique images (by hash)
        augment_count = 0

        while augment_count < num_augmentations:
            augmented_image = augment_image(image)

            # Use a hash to ensure uniqueness
            image_hash = hash(augmented_image.tobytes())
            if image_hash not in saved_images:
                saved_images.add(image_hash)
                augment_count += 1

                # Save the augmented image
                output_path = os.path.join(output_folder, f"{base_filename}_aug_{augment_count}.jpg")
                cv2.imwrite(output_path, augmented_image)

        print(f"Generated {num_augmentations} augmentations for {filename}")

# Run the function
process_images(input_folder, output_folder)


Generated 20 augmentations for china-1-yuan-2019 (1).jpg
Generated 20 augmentations for china-1-yuan-2019 (10).jpg
Generated 20 augmentations for china-1-yuan-2019 (11).jpg
Generated 20 augmentations for china-1-yuan-2019 (12).jpg
Generated 20 augmentations for china-1-yuan-2019 (13).jpg
Generated 20 augmentations for china-1-yuan-2019 (14).jpg
Generated 20 augmentations for china-1-yuan-2019 (15).jpg
Generated 20 augmentations for china-1-yuan-2019 (16).jpg
Generated 20 augmentations for china-1-yuan-2019 (17).jpg
Generated 20 augmentations for china-1-yuan-2019 (18).jpg
Generated 20 augmentations for china-1-yuan-2019 (19).jpg
Generated 20 augmentations for china-1-yuan-2019 (2).jpg
Generated 20 augmentations for china-1-yuan-2019 (20).jpg
Generated 20 augmentations for china-1-yuan-2019 (21).jpg
Generated 20 augmentations for china-1-yuan-2019 (22).jpg
Generated 20 augmentations for china-1-yuan-2019 (23).jpg
Generated 20 augmentations for china-1-yuan-2019 (3).jpg
Generated 20 augm

In [5]:
import os
import cv2
import numpy as np
from random import randint, uniform, random, seed

# ===============================
# Reproducibility
# ===============================
seed(42)
np.random.seed(42)

# ===============================
# Folder Paths
# ===============================
input_folder = "E:/Coins/New_Dataset_V2/train/Korean One Won Coin"
output_folder = "E:/Coins/New_Dataset_V2/one"
target_total_images = 3000

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# ===============================
# Augmentation Function
# ===============================
def augment_image(image):

    height, width = image.shape[:2]

    # 1️⃣ Random Resize
    scale = uniform(0.9, 1.1)
    new_w = int(width * scale)
    new_h = int(height * scale)

    # Prevent too small images
    new_w = max(new_w, 64)
    new_h = max(new_h, 64)

    image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    # 2️⃣ Full 360° Rotation
    angle = randint(0, 359)
    center = (new_w // 2, new_h // 2)

    rot_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    image = cv2.warpAffine(
        image,
        rot_matrix,
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT
    )

    # 3️⃣ Random Translation
    if random() < 0.4:
        max_tx = int(0.15 * new_w)
        max_ty = int(0.15 * new_h)

        tx = randint(-max_tx, max_tx)
        ty = randint(-max_ty, max_ty)

        trans_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
        image = cv2.warpAffine(
            image,
            trans_matrix,
            (new_w, new_h),
            borderMode=cv2.BORDER_REFLECT
        )

    # 4️⃣ Mild Perspective
    if random() < 0.3:
        margin = int(0.08 * min(new_w, new_h))

        src = np.float32([
            [randint(0, margin), randint(0, margin)],
            [new_w - randint(0, margin), randint(0, margin)],
            [randint(0, margin), new_h - randint(0, margin)],
            [new_w - randint(0, margin), new_h - randint(0, margin)]
        ])

        dst = np.float32([
            [0, 0],
            [new_w, 0],
            [0, new_h],
            [new_w, new_h]
        ])

        matrix = cv2.getPerspectiveTransform(src, dst)
        image = cv2.warpPerspective(
            image,
            matrix,
            (new_w, new_h),
            borderMode=cv2.BORDER_REFLECT
        )

    # 5️⃣ Random Crop + Resize
    if random() < 0.5:
        crop_scale = uniform(0.85, 1.0)
        crop_w = int(new_w * crop_scale)
        crop_h = int(new_h * crop_scale)

        if crop_w < new_w and crop_h < new_h:
            x = randint(0, new_w - crop_w)
            y = randint(0, new_h - crop_h)

            cropped = image[y:y + crop_h, x:x + crop_w]
            image = cv2.resize(cropped, (new_w, new_h))

    # 6️⃣ Blur (safe odd kernel only)
    if random() < 0.3:
        k = np.random.choice([3, 5])
        image = cv2.GaussianBlur(image, (k, k), 0)

    # 7️⃣ Add Gaussian Noise
    if random() < 0.3:
        noise = np.random.normal(0, 8, image.shape).astype(np.float32)
        image = np.clip(image.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # 8️⃣ Brightness & Contrast
    brightness = uniform(-25, 25)
    contrast = uniform(0.85, 1.15)
    image = np.clip(image * contrast + brightness, 0, 255).astype(np.uint8)

    return image


# ===============================
# Main Generator Function
# ===============================
def generate_until_target(input_folder, output_folder, target_total):

    source_images = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]

    if len(source_images) == 0:
        print("No valid images found in input folder.")
        return

    existing_images = [
        f for f in os.listdir(output_folder)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]

    current_count = len(existing_images)
    print(f"Currently {current_count} images in output folder.")

    if current_count >= target_total:
        print("Target already reached.")
        return

    saved_hashes = set()

    while current_count < target_total:

        filename = source_images[randint(0, len(source_images) - 1)]
        input_path = os.path.join(input_folder, filename)

        image = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            continue

        aug_img = augment_image(image)
        img_hash = hash(aug_img.tobytes())

        if img_hash in saved_hashes:
            continue

        saved_hashes.add(img_hash)

        save_path = os.path.join(
            output_folder,
            f"aug_{current_count + 1}.jpg"
        )

        cv2.imwrite(save_path, aug_img)
        current_count += 1

        if current_count % 100 == 0:
            print(f"{current_count} images generated...")

    print("Finished. Target of 3000 images reached.")


# ===============================
# Run
# ===============================
generate_until_target(input_folder, output_folder, target_total_images)

Currently 0 images in output folder.
100 images generated...
200 images generated...
300 images generated...
400 images generated...
500 images generated...
600 images generated...
700 images generated...
800 images generated...
900 images generated...
1000 images generated...
1100 images generated...
1200 images generated...
1300 images generated...
1400 images generated...
1500 images generated...
1600 images generated...
1700 images generated...
1800 images generated...
1900 images generated...
2000 images generated...
2100 images generated...
2200 images generated...
2300 images generated...
2400 images generated...
2500 images generated...
2600 images generated...
2700 images generated...
2800 images generated...
2900 images generated...
3000 images generated...
Finished. Target of 3000 images reached.
